SVD LOW RANK FACTO FOR YOLO

In [ ]:
import argparse
import math
import os
from copy import deepcopy
from datetime import datetime
from pathlib import Path
from typing import List, Union

import numpy as np
import torch
import torch.nn as nn
from matplotlib import pyplot as plt
from ultralytics import YOLO, __version__
from ultralytics.nn.modules import Detect, C2f, Conv, Bottleneck
from ultralytics.engine.trainer import BaseTrainer
from ultralytics.utils import YAML, LOGGER, RANK, DEFAULT_CFG_DICT, DEFAULT_CFG_KEYS
from ultralytics.utils.checks import check_yaml
from ultralytics.utils.torch_utils import initialize_weights
import torch_pruning as tp

print("✅ All imports successful!")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"torch_pruning version: {tp.__version__}")

def patch_torch_load_for_pytorch26():
    """Patch torch.load to work with PyTorch 2.6+ security changes"""
    original_load = torch.load
    
    def patched_load(f, map_location=None, weights_only=None, **kwargs):
        try:
            return original_load(f, map_location=map_location, weights_only=False, **kwargs)
        except Exception as e:
            return original_load(f, map_location=map_location, weights_only=weights_only, **kwargs)
    
    torch.load = patched_load
    print("✅ torch.load patched for PyTorch 2.6+ compatibility")

patch_torch_load_for_pytorch26()



def de_parallel(model):
    """Remove DataParallel wrapper if present"""
    return model.module if isinstance(model, nn.DataParallel) else model


def attempt_load_one_weight(model_path, device='cpu'):
    """Load model with compatibility for different ultralytics versions"""
    try:
        from ultralytics import YOLO as YOLOModel
        model = YOLOModel(model_path)
        return model.model, None
    except Exception as e:
        print(f"❌ Could not load model: {e}")
        return None, None


def ensure_loss_device_compatibility():
    """Ensure Ultralytics v8DetectionLoss proj tensor is on the correct device."""
    try:
        from ultralytics.utils.loss import v8DetectionLoss
    except Exception:
        return

    if getattr(v8DetectionLoss, "_device_compatibility_wrapped", False):
        return

    orig_bbox_decode = v8DetectionLoss.bbox_decode

    def bbox_decode_fixed(self, anchor_points, pred_dist):
        if hasattr(self, 'proj') and isinstance(self.proj, torch.Tensor):
            if self.proj.device != pred_dist.device:
                self.proj = self.proj.to(pred_dist.device)
        return orig_bbox_decode(self, anchor_points, pred_dist)

    v8DetectionLoss.bbox_decode = bbox_decode_fixed
    v8DetectionLoss._device_compatibility_wrapped = True


def save_pruning_performance_graph(x, y1, y11, y2, pruning_method="L2"):
    """Draw performance change graph"""
    try:
        plt.style.use("ggplot")
    except:
        pass

    x, y1, y11, y2 = np.array(x), np.array(y1), np.array(y11), np.array(y2)
    y2_ratio = y2 / y2[0]

    fig, ax = plt.subplots(figsize=(12, 8))
    ax.set_xlabel('Pruning Ratio (%)')
    ax.set_ylabel('mAP')
    ax.plot(x, y1, label='recovered mAP', marker='o')
    ax.plot(x, y11, color='tab:blue', label='recovered mAP50', marker='s')

    ax2 = ax.twinx()
    ax2.set_ylabel('MACs Ratio')
    ax2.plot(x, y2_ratio, color='tab:orange', label='MACs', marker='^')

    lines, labels = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax2.legend(lines + lines2, labels + labels2, loc='best')

    ax.set_xlim(105, 20)
    ax.set_ylim(0, max(y11) + 0.05)
    ax2.set_ylim(0.05, 1.05)

    max_y1_idx = np.argmax(y1)
    max_y11_idx = np.argmax(y11)

    ax.text(x[max_y1_idx], y1[max_y1_idx] + 0.02, f'max mAP = {y1[max_y1_idx]:.3f}', fontsize=10)
    ax.text(x[max_y11_idx], y11[max_y11_idx] + 0.02, f'max mAP50 = {y11[max_y11_idx]:.3f}', fontsize=10)

    plt.title('mAP / mAP50 and MACs vs Pruning Ratio')
    os.makedirs('results', exist_ok=True)
    plt.savefig(f'results/pruning_perf_change_{pruning_method}.png', dpi=150, bbox_inches='tight')
    print(f"✅ Performance graph saved: results/pruning_perf_change_{pruning_method}.png")


def infer_shortcut(bottleneck):
    c1 = bottleneck.cv1.conv.in_channels
    c2 = bottleneck.cv2.conv.out_channels
    return c1 == c2 and hasattr(bottleneck, 'add') and bottleneck.add


print("✅ Helper functions defined!")


class C2f_v2(nn.Module):
    """CSP Bottleneck with 2 convolutions - Modified for pruning"""
    def __init__(self, c1, c2, n=1, shortcut=False, g=1, e=0.5):
        super().__init__()
        self.c = int(c2 * e)
        self.cv0 = Conv(c1, self.c, 1, 1)
        self.cv1 = Conv(c1, self.c, 1, 1)
        self.cv2 = Conv((2 + n) * self.c, c2, 1)
        self.m = nn.ModuleList(Bottleneck(self.c, self.c, shortcut, g, k=((3, 3), (3, 3)), e=1.0) for _ in range(n))

    def forward(self, x):
        y = [self.cv0(x), self.cv1(x)]
        y.extend(m(y[-1]) for m in self.m)
        return self.cv2(torch.cat(y, 1))


def transfer_weights(c2f, c2f_v2):
    """Transfer weights from C2f to C2f_v2"""
    c2f_v2.cv2 = c2f.cv2
    c2f_v2.m = c2f.m

    state_dict = c2f.state_dict()
    state_dict_v2 = c2f_v2.state_dict()

    old_weight = state_dict['cv1.conv.weight']
    half_channels = old_weight.shape[0] // 2
    state_dict_v2['cv0.conv.weight'] = old_weight[:half_channels]
    state_dict_v2['cv1.conv.weight'] = old_weight[half_channels:]

    for bn_key in ['weight', 'bias', 'running_mean', 'running_var']:
        old_bn = state_dict[f'cv1.bn.{bn_key}']
        state_dict_v2[f'cv0.bn.{bn_key}'] = old_bn[:half_channels]
        state_dict_v2[f'cv1.bn.{bn_key}'] = old_bn[half_channels:]

    for key in state_dict:
        if not key.startswith('cv1.'):
            state_dict_v2[key] = state_dict[key]

    for attr_name in dir(c2f):
        attr_value = getattr(c2f, attr_name)
        if not callable(attr_value) and '_' not in attr_name:
            setattr(c2f_v2, attr_name, attr_value)

    c2f_v2.load_state_dict(state_dict_v2)


def replace_c2f_with_c2f_v2(module):
    """Replace all C2f modules with C2f_v2"""
    for name, child_module in module.named_children():
        if isinstance(child_module, C2f):
            shortcut = infer_shortcut(child_module.m[0])
            c2f_v2 = C2f_v2(child_module.cv1.conv.in_channels, child_module.cv2.conv.out_channels,
                            n=len(child_module.m), shortcut=shortcut,
                            g=child_module.m[0].cv2.conv.groups,
                            e=child_module.c / child_module.cv2.conv.out_channels)
            transfer_weights(child_module, c2f_v2)
            setattr(module, name, c2f_v2)
        else:
            replace_c2f_with_c2f_v2(child_module)


print(" C2f_v2 module defined!")



def save_model_v2(self: BaseTrainer):
    """Disabled half precision saving"""
    ckpt = {
        'epoch': self.epoch,
        'best_fitness': self.best_fitness,
        'model': deepcopy(de_parallel(self.model)),
        'ema': deepcopy(self.ema.ema),
        'updates': self.ema.updates,
        'optimizer': self.optimizer.state_dict(),
        'train_args': vars(self.args),
        'date': datetime.now().isoformat(),
        'version': __version__}

    torch.save(ckpt, self.last)
    if self.best_fitness == self.fitness:
        torch.save(ckpt, self.best)
    if (self.epoch > 0) and (self.save_period > 0) and (self.epoch % self.save_period == 0):
        torch.save(ckpt, self.wdir / f'epoch{self.epoch}.pt')
    del ckpt


def final_eval_v2(self: BaseTrainer):
    """Modified final evaluation"""
    for f in self.last, self.best:
        if f.exists():
            strip_optimizer_v2(f)
            if f is self.best:
                LOGGER.info(f'\nValidating {f}...')
                self.metrics = self.validator(model=f)
                self.metrics.pop('fitness', None)
                self.run_callbacks('on_fit_epoch_end')


def strip_optimizer_v2(f: Union[str, Path] = 'best.pt', s: str = '') -> None:
    """Strip optimizer from checkpoint"""
    x = torch.load(f, map_location=torch.device('cpu'))
    args = {**DEFAULT_CFG_DICT, **x['train_args']}
    if x.get('ema'):
        x['model'] = x['ema']
    for k in 'optimizer', 'ema', 'updates':
        x[k] = None
    for p in x['model'].parameters():
        p.requires_grad = False
    x['train_args'] = {k: v for k, v in args.items() if k in DEFAULT_CFG_KEYS}
    torch.save(x, s or f)
    mb = os.path.getsize(s or f) / 1E6
    LOGGER.info(f"Optimizer stripped from {f}, {mb:.1f}MB")


def train_v2(self: YOLO, pruning=False, **kwargs):
    """Modified training function for pruning"""
    self._check_is_pytorch_model()
    overrides = self.overrides.copy()
    overrides.update(kwargs)
    if kwargs.get('cfg'):
        LOGGER.info(f"cfg file passed. Overriding default params with {kwargs['cfg']}.")
        overrides = YAML.load(check_yaml(kwargs['cfg']))
    overrides['mode'] = 'train'
    if not overrides.get('data'):
        raise AttributeError("Dataset required but missing, i.e. pass 'data=coco128.yaml'")
    if overrides.get('resume'):
        overrides['resume'] = self.ckpt_path

    self.task = overrides.get('task') or self.task
    self.trainer = self.task_map[self.task]['trainer'](overrides=overrides, _callbacks=self.callbacks)

    if not pruning:
        if not overrides.get('resume'):
            self.trainer.model = self.trainer.get_model(weights=self.model if self.ckpt else None, cfg=self.model.yaml)
            self.model = self.trainer.model
    else:
        self.trainer.pruning = True
        self.trainer.model = self.model
        self.trainer.save_model = save_model_v2.__get__(self.trainer)
        self.trainer.final_eval = final_eval_v2.__get__(self.trainer)

    ensure_loss_device_compatibility()
    self.trainer.hub_session = self.session
    self.trainer.train()
    
    if RANK in (-1, 0):
        try:
            active_device = next(self.model.parameters()).device
        except Exception:
            active_device = None
        loaded_model, _ = attempt_load_one_weight(str(self.trainer.best), device=active_device)
        if loaded_model is not None:
            self.model = loaded_model
        self.overrides = self.model.args
        self.metrics = getattr(self.trainer.validator, 'metrics', None)


print("✅ Training override functions defined!")

c:\Users\alijo\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ All imports successful!
PyTorch: 2.6.0+cu124
CUDA available: True
torch_pruning version: 1.6.0
✅ torch.load patched for PyTorch 2.6+ compatibility
✅ Helper functions defined!
✅ C2f_v2 module defined!
✅ Training override functions defined!


In [ ]:
import torch
import torch.nn as nn
from ultralytics import YOLO
from pathlib import Path

# ================== UPDATE THESE PATHS ==================
WEIGHTS_PATH = r"C:\Users\alijo\OneDrive\Desktop\BachelorThesis\BachelorThesis\Models\CombinedDS\YOLOV8N\weights\best.pt"
DATA_YAML = r"C:\Users\alijo\OneDrive\Desktop\BachelorThesis\BachelorThesis\MixedDS.yaml"

IMG_SIZE = 640
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RANK_RATIO = 0.1  # Use 50% rank (try 0.3 for more compression)
# ========================================================


def decompose_conv_svd(conv_layer, rank_ratio=0.5):
    """
    Decompose Conv2d using SVD: Conv(in, out, k, k) → Conv(in, r, k, k) + Conv(r, out, 1, 1)
    """
    weight = conv_layer.weight.data
    out_ch, in_ch, kh, kw = weight.shape

    if (kh == 1 and kw == 1) or min(out_ch, in_ch) < 8:
        return None, None, 0
 
    max_rank = min(out_ch, in_ch * kh * kw)
    rank = max(1, int(max_rank * rank_ratio))

    orig_params = out_ch * in_ch * kh * kw
    new_params = (rank * in_ch * kh * kw) + (out_ch * rank * 1 * 1)
    
    if new_params >= orig_params * 0.95:  
        return None, None, 0
    

    weight_2d = weight.reshape(out_ch, in_ch * kh * kw)
    U, S, V = torch.svd(weight_2d)

    U_r = U[:, :rank]
    S_r = S[:rank]
    V_r = V[:, :rank]
    
    first_weight = (V_r.T * S_r.unsqueeze(1)).reshape(rank, in_ch, kh, kw)
    first_conv = nn.Conv2d(
        in_ch, rank, (kh, kw),
        stride=conv_layer.stride,
        padding=conv_layer.padding,
        bias=False
    ).to(weight.device)
    first_conv.weight.data = first_weight
    
    second_weight = U_r.T.reshape(rank, out_ch, 1, 1).permute(1, 0, 2, 3)
    second_conv = nn.Conv2d(
        rank, out_ch, (1, 1),
        bias=conv_layer.bias is not None
    ).to(weight.device)
    second_conv.weight.data = second_weight
    
    if conv_layer.bias is not None:
        second_conv.bias.data = conv_layer.bias.data.clone()
    
    return first_conv, second_conv, orig_params - new_params


def apply_lowrank_to_model(model, rank_ratio=0.5):
    """
    Apply low-rank factorization to all suitable Conv2d layers
    """
    print(f"\nApplying low-rank factorization (rank_ratio={rank_ratio})...")
    
    total_reduction = 0
    decomposed = 0

    conv_layers = []
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            conv_layers.append((name, module))
    
    print(f"Found {len(conv_layers)} Conv2d layers")

    for name, conv in conv_layers:
        first, second, reduction = decompose_conv_svd(conv, rank_ratio)
        
        if first is not None:
            parts = name.split('.')
            parent = model
            for part in parts[:-1]:
                parent = getattr(parent, part)
            
            setattr(parent, parts[-1], nn.Sequential(first, second))
            
            total_reduction += reduction
            decomposed += 1
            
            if decomposed <= 10: 
                print(f"  ✓ {name}: -{reduction:,} params")
    
    print(f"\nDecomposed {decomposed}/{len(conv_layers)} layers")
    print(f"Total param reduction: {total_reduction:,}")
    
    return total_reduction


# ==================== MAIN ====================

print("="*70)
print("LOW-RANK FACTORIZATION FOR YOLO")
print("="*70)
print("\n[1/5] Loading model...")
yolo = YOLO(WEIGHTS_PATH)
model = yolo.model.to(DEVICE)

orig_params = sum(p.numel() for p in model.parameters())
print(f"Original: {orig_params:,} ({orig_params/1e6:.2f}M params)")
print("\n[3/5] Applying factorization...")
param_reduction = apply_lowrank_to_model(model, RANK_RATIO)

new_params = sum(p.numel() for p in model.parameters())
reduction_pct = (1 - new_params/orig_params) * 100

print(f"\nAfter factorization:")
print(f"  New params: {new_params:,} ({new_params/1e6:.2f}M)")
print(f"  Reduction:  {reduction_pct:.1f}%")

print("\n[4/5] Testing model...")
try:
    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
    with torch.no_grad():
        _ = model(dummy)
    print("✓ Forward pass works!")
except Exception as e:
    print(f"❌ Forward pass failed: {e}")
    exit()

print("\n[5/5] Evaluating factorized model...")
yolo.model = model
out_path = Path(WEIGHTS_PATH).parent / "best_lowrank.pt"
torch.save(model.state_dict(), out_path)
print(f"\n✓ Saved: {out_path}")
print("\n" + "="*70)
print("RESULTS")
print("="*70)
print(f"\n{'Metric':<25} {'Original':<15} {'Factorized':<15}")
print("-" * 55)
print(f"{'Parameters':<25} {f'{orig_params/1e6:.2f}M':<15} {f'{new_params/1e6:.2f}M':<15}")
print(f"{'Reduction':<25} {'-':<15} {f'{reduction_pct:.1f}%':<15}")

LOW-RANK FACTORIZATION FOR YOLO

[1/5] Loading model...
Original: 3,011,043 (3.01M params)

[2/5] Baseline evaluation...
Ultralytics 8.3.223  Python-3.10.11 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 391.5146.7 MB/s, size: 64.1 KB)
val: Scanning C:\Users\alijo\OneDrive\Desktop\LeafminerOnly\test\labels.cache... 197 images, 60 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 197/197 198.5Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 4.9it/s 2.7s0.1s
                   all        197        182      0.937      0.874      0.941       0.83
Speed: 2.0ms preprocess, 4.2ms inference, 0.0ms loss, 2.8ms postprocess per image
Results saved to C:\Users\alijo\OneDrive\Desktop\BachelorThesis\BachelorThesis\Optimization\runs\detect\val3
Baseline mAP50-95: 0.8304

[3/5] Applyi

In [ ]:
import os
import inspect
import numpy as np
import torch
import torch.nn as nn

from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer

import ultralytics.utils.tal as tal_mod
import ultralytics.utils.loss as loss_mod
import ultralytics.data.build as build_mod


# ================== CONFIGURATION ==================
WEIGHTS_PATH = r"C:\Users\alijo\OneDrive\Desktop\BachelorThesis\BachelorThesis\Optimization\YOLO_V5_BN_0.8\step_0_finetune\weights\best.pt"
DATA_YAML = r"C:\Users\alijo\OneDrive\Desktop\BachelorThesis\BachelorThesis\MixedDS.yaml"
IMG_SIZE = 320
RANK_RATIO = 0.12
FINETUNE_EPOCHS = 100
BATCH = 16
LR0 = 0.001
OPTIMIZER = "SGD"
AMP = True
VAL = False  
# ===================================================


def pick_device():
    if torch.cuda.is_available() and torch.cuda.device_count() > 0:
        torch.cuda.set_device(0)
        return "cuda:0", 0
    return "cpu", "cpu"


DEVICE, ULTRA_DEVICE = pick_device()
torch.backends.cudnn.benchmark = True if str(DEVICE).startswith("cuda") else False


def patch_ultralytics_build_dataloader_no_pin():
    """
    Your crash is inside torch DataLoader pin_memory().
    Ultralytics enables pin_memory internally, so we patch the function that builds dataloaders
    to ALWAYS set pin_memory=False.
    """
    if not hasattr(build_mod, "build_dataloader"):
        raise RuntimeError("ultralytics.data.build.build_dataloader not found; can't patch pin_memory.")

    orig = build_mod.build_dataloader

    def patched_build_dataloader(*args, **kwargs):
        kwargs["pin_memory"] = False 
        kwargs["workers"] = 0        
        return orig(*args, **kwargs)

    build_mod.build_dataloader = patched_build_dataloader
    print("✅ Patched ultralytics.data.build.build_dataloader(): pin_memory=False, workers=0")


def patch_task_aligned_assigner_forward_device():
    orig_forward = tal_mod.TaskAlignedAssigner.forward
    sig = list(inspect.signature(orig_forward).parameters.keys())
    if not sig or sig[0] != "self":
        raise RuntimeError("Unexpected TaskAlignedAssigner.forward signature; can't patch safely.")

    def patched_forward(self, pd_scores, pd_bboxes, anc_points, gt_labels, gt_bboxes, mask_gt):
        dev = pd_scores.device
        if torch.is_tensor(pd_bboxes) and pd_bboxes.device != dev:
            pd_bboxes = pd_bboxes.to(dev, non_blocking=True)
        if torch.is_tensor(anc_points) and anc_points.device != dev:
            anc_points = anc_points.to(dev, non_blocking=True)
        if torch.is_tensor(gt_labels) and gt_labels.device != dev:
            gt_labels = gt_labels.to(dev, non_blocking=True)
        if torch.is_tensor(gt_bboxes) and gt_bboxes.device != dev:
            gt_bboxes = gt_bboxes.to(dev, non_blocking=True)
        if torch.is_tensor(mask_gt) and mask_gt.device != dev:
            mask_gt = mask_gt.to(dev, non_blocking=True)
        return orig_forward(self, pd_scores, pd_bboxes, anc_points, gt_labels, gt_bboxes, mask_gt)

    tal_mod.TaskAlignedAssigner.forward = patched_forward
    print("✅ Patched TaskAlignedAssigner.forward(): force inputs to pd_scores.device")


# HARD PATCH C: FIX v8DetectionLoss.proj DEVICE 
def patch_v8_loss_bbox_decode_proj_device():
    """
    Fixes crash in bbox_decode matmul when self.proj lives on CPU but pred_dist is CUDA.
    """
    if not hasattr(loss_mod, "v8DetectionLoss"):
        print("⚠️ v8DetectionLoss not found; skipping proj patch.")
        return

    orig_bbox_decode = loss_mod.v8DetectionLoss.bbox_decode

    def patched_bbox_decode(self, anchor_points, pred_dist):
        if hasattr(self, "proj") and torch.is_tensor(self.proj) and self.proj.device != pred_dist.device:
            self.proj = self.proj.to(pred_dist.device)
        return orig_bbox_decode(self, anchor_points, pred_dist)

    loss_mod.v8DetectionLoss.bbox_decode = patched_bbox_decode
    print("✅ Patched v8DetectionLoss.bbox_decode(): force self.proj onto pred_dist.device")


def deep_to_device(x, device: torch.device):
    if torch.is_tensor(x):
        return x.to(device, non_blocking=True)
    if isinstance(x, np.ndarray):
        return torch.from_numpy(x).to(device, non_blocking=True)
    if isinstance(x, dict):
        return {k: deep_to_device(v, device) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        out = [deep_to_device(v, device) for v in x]
        return type(x)(out)
    return x


def patch_trainer_preprocess_to_device(trainer: DetectionTrainer):
    orig = trainer.preprocess_batch

    def preprocess_batch_force_device(batch):
        batch = orig(batch)
        return deep_to_device(batch, trainer.device)

    trainer.preprocess_batch = preprocess_batch_force_device
    print("✅ Patched trainer.preprocess_batch(): deep-move batch to trainer.device")


def decompose_conv_svd(conv_layer: nn.Conv2d, rank_ratio: float = 0.5):
    weight = conv_layer.weight.data
    out_ch, in_ch, kh, kw = weight.shape
    if (kh == 1 and kw == 1) or min(out_ch, in_ch) < 8:
        return None, None, 0

    max_rank = min(out_ch, in_ch * kh * kw)
    rank = max(1, int(max_rank * rank_ratio))

    orig_params = out_ch * in_ch * kh * kw
    new_params = (rank * in_ch * kh * kw) + (out_ch * rank)
    if new_params >= orig_params * 0.95:
        return None, None, 0

    weight_2d = weight.reshape(out_ch, in_ch * kh * kw)
    U, S, V = torch.svd(weight_2d)

    U_r = U[:, :rank]
    S_r = S[:rank]
    V_r = V[:, :rank]

    first_weight = (V_r.T * S_r.unsqueeze(1)).reshape(rank, in_ch, kh, kw)
    first_conv = nn.Conv2d(
        in_channels=in_ch,
        out_channels=rank,
        kernel_size=(kh, kw),
        stride=conv_layer.stride,
        padding=conv_layer.padding,
        dilation=conv_layer.dilation,
        groups=conv_layer.groups,
        bias=False,
        padding_mode=conv_layer.padding_mode,
    )
    first_conv.weight.data.copy_(first_weight)

    second_weight = U_r.reshape(out_ch, rank, 1, 1)
    second_conv = nn.Conv2d(
        in_channels=rank,
        out_channels=out_ch,
        kernel_size=(1, 1),
        stride=1,
        padding=0,
        bias=conv_layer.bias is not None,
    )
    second_conv.weight.data.copy_(second_weight)
    if conv_layer.bias is not None:
        second_conv.bias.data.copy_(conv_layer.bias.data)

    return first_conv, second_conv, (orig_params - new_params)


def _set_module_by_name(root: nn.Module, name: str, new_module: nn.Module):
    parts = name.split(".")
    parent = root
    for p in parts[:-1]:
        parent = getattr(parent, p)
    setattr(parent, parts[-1], new_module)


def apply_lowrank_to_model(model: nn.Module, rank_ratio: float):
    print(f"\nApplying SVD (rank_ratio={rank_ratio})...")
    total_reduction, replaced = 0, 0
    conv_targets = [(n, m) for n, m in model.named_modules() if isinstance(m, nn.Conv2d)]
    for name, conv in conv_targets:
        first, second, reduction = decompose_conv_svd(conv, rank_ratio)
        if first is not None:
            _set_module_by_name(model, name, nn.Sequential(first, second))
            total_reduction += reduction
            replaced += 1
    print(f"Replaced convs: {replaced}")
    print(f"Estimated param reduction: {total_reduction:,}")
    return total_reduction


def ensure_model_on_device(m: nn.Module, device: torch.device):
    m.to(device)
    for p in m.parameters():
        if p.data.device != device:
            p.data = p.data.to(device)
    for b in m.buffers():
        if b.data.device != device:
            b.data = b.data.to(device)
    return m


def move_loss_modules_to_device(model: nn.Module, device: torch.device):
    crit = getattr(model, "criterion", None)
    if isinstance(crit, nn.Module):
        crit.to(device)
        assigner = getattr(crit, "assigner", None)
        if isinstance(assigner, nn.Module):
            assigner.to(device)


def main():
    print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
    print(f"Using DEVICE = {DEVICE} | ULTRA_DEVICE = {ULTRA_DEVICE}")
    patch_ultralytics_build_dataloader_no_pin()
    patch_task_aligned_assigner_forward_device()
    patch_v8_loss_bbox_decode_proj_device()

    print(f"\nLoading YOLO model from: {WEIGHTS_PATH}")
    yolo = YOLO(WEIGHTS_PATH)
    model = yolo.model

    model = ensure_model_on_device(model, torch.device(DEVICE))
    apply_lowrank_to_model(model, RANK_RATIO)
    model = ensure_model_on_device(model, torch.device(DEVICE))

    target_params = sum(p.numel() for p in model.parameters())
    print(f"SVD model params: {target_params:,}")

    custom_args = dict(
        model=WEIGHTS_PATH,
        data=DATA_YAML,
        epochs=FINETUNE_EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH,
        device=ULTRA_DEVICE,
        lr0=LR0,
        optimizer=OPTIMIZER,
        pretrained=False,
        amp=AMP,
        verbose=True,
        val=VAL,        
        workers=0,      
    )

    trainer = DetectionTrainer(overrides=custom_args)

    trainer.device = torch.device(DEVICE)
    trainer.model = model.train().to(trainer.device)

    move_loss_modules_to_device(trainer.model, trainer.device)
    patch_trainer_preprocess_to_device(trainer)

    print("\nSTARTING TRAINING")
    print("trainer.device:", trainer.device)
    trainer.train()

    print("\nRun directory:", trainer.save_dir)
    print("Best weights:", trainer.save_dir / "weights" / "best.pt")
    print("Last weights:", trainer.save_dir / "weights" / "last.pt")


if __name__ == "__main__":
    main()


torch.cuda.is_available(): True
Using DEVICE = cuda:0 | ULTRA_DEVICE = 0
✅ Patched ultralytics.data.build.build_dataloader(): pin_memory=False, workers=0
✅ Patched TaskAlignedAssigner.forward(): force inputs to pd_scores.device
✅ Patched v8DetectionLoss.bbox_decode(): force self.proj onto pred_dist.device

Loading YOLO model from: C:\Users\alijo\OneDrive\Desktop\BachelorThesis\BachelorThesis\Optimization\YOLO_V5_BN_0.8\step_0_finetune\weights\best.pt

Applying SVD (rank_ratio=0.12)...
Replaced convs: 27
Estimated param reduction: 477,805
SVD model params: 141,239
Ultralytics 8.3.223  Python-3.10.11 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\alijo\OneDrive\Desktop\BachelorThesis\Bac

KeyboardInterrupt: 

In [91]:
from ultralytics import YOLO
m = YOLO(r"C:\Users\alijo\OneDrive\Desktop\BachelorThesis\BachelorThesis\Optimization\runs\detect\BN_SVD_0.9\weights\best.pt")
print(sum(p.numel() for p in m.model.parameters()))


911098


In [ ]:
from ultralytics import YOLO
import torch

hiha = YOLO(r"C:\Users\alijo\OneDrive\Desktop\BachelorThesis\BachelorThesis\Optimization\YOLO_pruning_ratio_90_Random\ratio_89_step_0_finetune\weights\best.pt")


hiha.model.fuse = lambda verbose=True: hiha.model
hiha.model.info(verbose=False)  

total_params = sum(p.numel() for p in hiha.model.parameters())
trainable_params = sum(p.numel() for p in hiha.model.parameters() if p.requires_grad)
print(f"Total params: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

metrics = hiha.val(split="test", data=DATA_YAML,imgsz=340)
print(metrics)

Total params: 319,415
Trainable params: 0
WARNING imgsz=[340] must be multiple of max stride 32, updating to [352]
Ultralytics 8.3.223  Python-3.10.11 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
val: Fast image access  (ping: 0.10.0 ms, read: 242.351.7 MB/s, size: 53.1 KB)
val: Scanning C:\Users\alijo\OneDrive\Desktop\LeafminerOnly\test\labels.cache... 197 images, 60 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 197/197 102.5Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 3.5it/s 3.7s0.2s
                   all        197        182      0.769      0.695       0.75      0.446
Speed: 2.4ms preprocess, 8.6ms inference, 0.0ms loss, 2.1ms postprocess per image
Results saved to C:\Users\alijo\OneDrive\Desktop\BachelorThesis\BachelorThesis\Optimization\runs\detect\val91
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metri